# Привет, pytorch

![img](https://pytorch.org/tutorials/_static/pytorch-logo-dark.svg)

__Этот блокнот__ научит вас использовать низкоуровневое ядро pytorch. Вы можете установить его [здесь](http://pytorch.org/). Для высокоуровневого интерфейса смотрите следующий блокнот.

__Pytorch ощущается__ иначе, чем tensorflow/theano почти на каждом уровне. TensorFlow заставляет ваш код жить одновременно в двух "мирах": символьные графы и реальные тензоры. Сначала вы объявляете символьный "рецепт" того, как получить от входов к выходам, затем подаете ему реальные мини-пакеты данных. В pytorch __есть только один мир__: все тензоры имеют числовое значение.

Вы вычисляете выходы на лету без предварительного объявления чего-либо. Код выглядит точно как в чистом numpy с одним исключением: pytorch вычисляет градиенты за вас. И может выполнять вычисления на GPU. И имеет множество предварительно реализованных строительных блоков для ваших нейронных сетей. [И еще несколько вещей.](https://medium.com/towards-data-science/pytorch-vs-tensorflow-spotting-the-difference-25c75777377b)

А теперь мы наконец замолчим и дадим pytorch говорить.

In [ ]:
# если работаете в colab, выполните это:
# !wget https://raw.githubusercontent.com/yandexdataschool/Practical_DL/fall19/week02_autodiff/notmnist.py -O notmnist.py

from __future__ import print_function
import numpy as np
import pandas as pd
import torch
print(torch.__version__)  # ничего страшного, если ваша версия отличается, главное чтобы она была 1.0 или новее

In [ ]:
# мир numpy

x = np.arange(16).reshape(4, 4)

print("X :\n%s\n" % x)
print("X.shape : %s\n" % (x.shape,))
print("добавить 5 :\n%s\n" % (x + 5))
print("X*X^T  :\n%s\n" % np.dot(x, x.T))
print("среднее по столбцам :\n%s\n" % (x.mean(axis=-1)))
print("накопительная сумма столбцов :\n%s\n" % (np.cumsum(x, axis=0)))

In [ ]:
# мир pytorch

x = np.arange(16).reshape(4, 4)

x = torch.tensor(x, dtype=torch.float32)  # или torch.arange(0,16).view(4,4)

print("X :\n%s" % x)
print("X.shape : %s\n" % (x.shape,))
print("добавить 5 :\n%s" % (x + 5))
print("X*X^T  :\n%s" % torch.matmul(x, x.transpose(1, 0)))  # короткая форма: x.mm(x.t())
print("среднее по столбцам :\n%s" % torch.mean(x, dim=-1))
print("накопительная сумма столбцов :\n%s" % torch.cumsum(x, dim=0))

## NumPy и Pytorch

Как вы можете заметить, pytorch позволяет вам работать примерно так же, как с numpy. Нет объявления графа, нет заполнителей, нет сессий. Это означает, что вы можете _увидеть числовое значение любого тензора в любой момент времени_. Отладка такого кода может выполняться путем вывода тензоров или использования любого инструмента отладки, который вы хотите (например, [gdb](https://wiki.python.org/moin/DebuggingWithGdb)).

Вы также могли заметить несколько новых названий методов и другой API. Так что нет, совместимости с numpy [пока](https://github.com/pytorch/pytorch/issues/2228) нет, и да, вам придется запомнить все названия снова. Приготовьтесь!

![img](http://i0.kym-cdn.com/entries/icons/original/000/017/886/download.jpg)

Например,
* Если что-то принимает список/кортеж осей в numpy, вы можете ожидать, что в pytorch это будет *args
  * `x.reshape([1,2,8]) -> x.view(1,2,8)`
* Вы должны заменить _axis_ на _dim_ в операциях типа mean или cumsum
  * `x.sum(axis=-1) -> x.sum(dim=-1)`
* большинство математических операций одинаковы, но типы и изменение формы отличаются
  * `x.astype('int64') -> x.type(torch.LongTensor)`

Чтобы помочь вам адаптироваться, есть [таблица](https://github.com/torch/torch7/wiki/Torch-for-Numpy-users), охватывающая большинство новых вещей. Также есть аккуратная [страница документации](http://pytorch.org/docs/master/).

Наконец, если вы застряли с технической проблемой, мы рекомендуем поискать на [форумах pytorch](https://discuss.pytorch.org/). Или просто погуглить, что обычно работает так же эффективно.

Если вы чувствуете, что почти сдаетесь, помните две вещи: __GPU__ и __бесплатные градиенты__. Кроме того, вы всегда можете вернуться к numpy с помощью x.numpy()

### Разминка: тригонометрическое плетение
_вдохновлено [этим постом](https://www.quora.com/What-are-the-most-interesting-equation-plots)_

Есть несколько простых математических функций с крутыми графиками. Например, рассмотрите эту:

$$ x(t) = t - 1.5 * cos( 15 t) $$
$$ y(t) = t - 1.5 * sin( 16 t) $$

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

t = torch.linspace(-10, 10, steps=10000)

# вычислите x(t) и y(t) как определено выше
x =  # ВАШ КОД
y =  # ВАШ КОД

plt.plot(x.numpy(), y.numpy())

если закончили рано, попробуйте изменить формулу и посмотреть, как это влияет на функцию

## Автоматические градиенты

Любой уважающий себя фреймворк глубокого обучения должен выполнять обратное распространение за вас. Torch обрабатывает это с помощью модуля `autograd`.

Общий процесс выглядит так:
* При создании тензора вы помечаете его как `requires_grad`:
    * __```torch.zeros(5, requires_grad=True)```__
    * torch.tensor(np.arange(5), dtype=torch.float32, requires_grad=True)
* Определяете некоторую дифференцируемую `loss = arbitrary_function(a)`
* Вызываете `loss.backward()`
* Градиенты теперь доступны как ```a.grads```

__Вот пример:__ давайте обучим линейную регрессию на ценах на жилье в Бостоне

In [ ]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]
plt.scatter(data[:, -1], target)

In [ ]:
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

x = torch.tensor(data[:, -1] / 10, dtype=torch.float32)
y = torch.tensor(target, dtype=torch.float32)

Градиенты теперь сохранены в `.grad` тех тензоров, которые их требуют.

In [ ]:
print("dL/dw = \n", w.grad)
print("dL/db = \n", b.grad)

Если вы вычисляете градиент от нескольких функций потерь, градиенты будут суммироваться в тензорах, поэтому полезно __обнулять градиенты__ между итерациями.

In [ ]:
from IPython.display import clear_output

for i in range(100):

    y_pred = w * x + b
    loss = torch.mean((y_pred - y)**2)
    loss.backward()

    with torch.no_grad():
        w.data = w - 0.05 * w.grad.data
        b.data = b - 0.05 * b.grad.data

        # zero gradients
        w.grad.zero_()
        b.grad.zero_()

    # the rest of code is just bells and whistles
    if (i + 1) % 5 == 0:
        clear_output(True)
        plt.scatter(x.data.numpy(), y.data.numpy())
        plt.scatter(x.data.numpy(), y_pred.data.numpy(),
                    color='orange', linewidth=5)
        plt.show()

        print("loss = ", loss.data.numpy())
        if loss.item() < 0.5:
            print("Done!")
            break

__Бонусное задание__: попробуйте реализовать и написать какую-нибудь нелинейную регрессию. Вы можете попробовать квадратичные признаки или тригонометрию, или простую нейронную сеть. Единственное отличие в том, что теперь у вас больше весов и более сложный `y_pred`.

# Высокоуровневый pytorch

До сих пор мы работали с низкоуровневым API torch. Хотя это абсолютно необходимо для любых пользовательских функций потерь или слоев, создание больших нейронных сетей на нем довольно неуклюже.

К счастью, есть также высокоуровневый интерфейс torch с предопределенными слоями, активациями и алгоритмами обучения.

Мы рассмотрим их, решая простую задачу распознавания изображений: классификацию букв на __"A"__ против __"B"__.

In [ ]:
from notmnist import load_notmnist  # if not found: remember to un-comment the first cell
X_train, y_train, X_test, y_test = load_notmnist(letters='AB')
X_train, X_test = X_train.reshape([-1, 784]), X_test.reshape([-1, 784])

print("Train size = %i, test_size = %i" % (len(X_train), len(X_test)))

In [ ]:
for i in [0, 1]:
    plt.subplot(1, 2, i + 1)
    plt.imshow(X_train[i].reshape([28, 28]))
    plt.title(str(y_train[i]))

Давайте начнем со слоев. Основная абстракция здесь - __`torch.nn.Module`__

In [ ]:
from torch import nn
import torch.nn.functional as F

print(nn.Module.__doc__)

Есть обширная библиотека популярных слоев и архитектур, уже построенных для вас.

Это задача бинарной классификации, поэтому мы обучим __Логистическую регрессию с сигмоидой__.
$$P(y_i | X_i) = \sigma(W \cdot X_i + b) ={ 1 \over {1+e^{- [W \cdot X_i + b]}} }$$

In [ ]:
# create a network that stacks layers on top of each other# создаем сеть, которая "складывает" слои друг на друга
model = nn.Sequential(
    nn.Linear(784, 1),   # добавляем слой с 784 входами и 1 выходом
    nn.Sigmoid()         # добавляем softmax активацию для вероятностей. Нормализуем по оси 1
)

# заметка: вы также можете добавить слои с помощью model.add_module('l1', ), все имена слоев должны быть уникальными


In [ ]:
print("Weight shapes:", [w.shape for w in model.parameters()])

In [ ]:
# create dummy data with 3 samples and 784 features
x = torch.tensor(X_train[:3], dtype=torch.float32)
y = torch.tensor(y_train[:3], dtype=torch.float32)

# compute outputs given inputs, both are tensors
y_predicted = model(x)[:, 0]

y_predicted  # display what we've got

Теперь давайте определим функцию потерь для нашей модели.

Естественным выбором является использование бинарной кросс-энтропии (она же logloss, отрицательное llh):
$$ L = {1 \over N} \underset{X_i,y_i} \sum - [  y_i \cdot log P(y_i | X_i) + (1-y_i) \cdot log (1-P(y_i | X_i)) ]$$
Ваша задача - реализовать кросс-энтропийные потери __вручную__ без использования `torch.nn.functional`.

In [ ]:
crossentropy =  # YOUR CODE

loss =  # YOUR CODE

assert tuple(crossentropy.size()) == (
    3,), "Crossentropy must be a vector with element per sample"
assert tuple(loss.size()) == tuple(
), "Loss must be scalar. Did you forget the mean/sum?"
assert loss.data.numpy() > 0, "Crossentropy must non-negative, zero only for perfect prediction"
assert loss.data.numpy() <= np.log(
    3), "Loss is too large even for untrained model. Please double-check it."

__Примечание:__ вы также можете найти кросс-энтропийные потери в `torch.nn.functional`, просто наберите __`F.<tab>`__. Однако она работает с сырыми логитами вместо вероятностей.

__Оптимизаторы Torch__

Когда мы обучали линейную регрессию выше, нам приходилось вручную выполнять .zero_() для градиентов на обоих наших тензорах. Представьте этот код для 50-слойной сети.

Опять же, чтобы код не становился грязным, есть модуль `torch.optim` с предварительно реализованными алгоритмами:

In [ ]:
opt = torch.optim.RMSprop(model.parameters(), lr=0.01)

# here's how it's used:
loss.backward()      # add new gradients
opt.step()           # change weights
opt.zero_grad()      # clear gradients

In [ ]:
# dispose of old tensors to avoid bugs later
del x, y, y_predicted, loss, y_pred

### Собираем все вместе

In [ ]:
# create network again just in case
model = nn.Sequential()
model.add_module('first', nn.Linear(784, 1))
model.add_module('second', nn.Sigmoid())

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
history = []

for i in range(100):

    # sample 256 random images
    ix = np.random.randint(0, len(X_train), 256)
    x_batch = torch.tensor(X_train[ix], dtype=torch.float32)
    y_batch = torch.tensor(y_train[ix], dtype=torch.float32)

    # predict probabilities
    y_predicted =  # YOUR CODE

    assert y_predicted.dim(
    ) == 1, "did you forget to select first column with [:, 0]"

    # compute loss, just like before
    loss =  # YOUR CODE

    # compute gradients
    <YOUR CODE >

    # Adam step
    <YOUR CODE >

    # clear gradients
    <YOUR CODE >

    history.append(loss.data.numpy())

    if i % 10 == 0:
        print("step #%i | mean loss = %.3f" % (i, np.mean(history[-10:])))

__Советы по отладке:__
* убедитесь, что ваша модель правильно предсказывает вероятности. Просто выведите их и посмотрите, что внутри.
* не забудьте знак _минус_ в функции потерь! Это ошибка, которую 99% людей делают в какой-то момент.
* убедитесь, что вы обнуляете градиенты после каждого шага. Серьезно:)
* В целом, сообщения об ошибках pytorch довольно полезны, прочитайте их, прежде чем гуглить.
* если вы видите nan/inf, выведите то, что происходит на каждой итерации, чтобы выяснить, где именно это происходит.
  * Если потери уменьшаются, а затем превращаются в nan на полпути, попробуйте меньшую скорость обучения. (Наша текущая формула потерь нестабильна).

### Оценка

Давайте посмотрим, как наша модель работает на тестовых данных

In [ ]:
# use your model to predict classes (0 or 1) for all test samples
predicted_y_test =  # YOUR CODE

assert isinstance(predicted_y_test, np.ndarray), "please return np array, not %s" % type(
    predicted_y_test)
assert predicted_y_test.shape == y_test.shape, "please predict one class for each test sample"
assert np.in1d(predicted_y_test, y_test).all(), "please predict class indexes"

accuracy = np.mean(predicted_y_test == y_test)

print("Test accuracy: %.5f" % accuracy)
assert accuracy > 0.95, "try training longer"

## Больше о pytorch:
* Использование torch на GPU и multi-GPU - [ссылка](http://pytorch.org/docs/master/notes/cuda.html)
* Больше учебников по pytorch - [ссылка](http://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html)
* Примеры Pytorch - репозиторий, который реализует множество крутых моделей глубокого обучения в pytorch - [ссылка](https://github.com/pytorch/examples)
* Практический pytorch - репозиторий, который реализует некоторые... другие крутые модели глубокого обучения... да, в pytorch - [ссылка](https://github.com/spro/practical-pytorch)
* И еще немного - [ссылка](https://www.reddit.com/r/pytorch/comments/6z0yeo/pytorch_and_pytorch_tricks_for_kaggle/)